# IoT Sensor Anomaly Detection - Master Notebook

**Project:** AI-Based IoT Sensor Anomaly Detection
**Sensor:** ESP32 + DHT11 (GPIO 4)
**Sampling:** 2000 ms interval
**Communication:** USB Serial @ 115200 baud
**Deployment:** Windows PC - Python handles ML inference & dashboard

---

## SECTION 0: Project Overview & Learning Objectives

### Project Goal
Build an end-to-end anomaly detection system for temperature and humidity sensor data. The system will:
1. Collect data from ESP32 + DHT11 via serial communication
2. Clean and preprocess the sensor data
3. Engineer meaningful features for anomaly detection
4. Train an Isolation Forest model on normal data only
5. Detect anomalies in real-time with a monitoring dashboard

### What You'll Learn
- **Data loading & validation** - Handle real-world sensor data with missing headers, duplicates, and gaps
- **Exploratory Data Analysis (EDA)** - Understand distributions, detect outliers, visualize time series
- **Data cleaning** - Remove exact duplicates, handle timestamp gaps, flag suspicious readings
- **Feature engineering** - Create change features, rolling statistics, session-aware rolling windows
- **Anomaly detection** - Isolation Forest (unsupervised, trains on normal data only)
- **Model evaluation** - Precision, Recall, F1 on labeled test set

### Dataset Summary
| Dataset | Rows | Purpose |
|---------|------|---------|
| normal_data.csv | 36,798 | Candidate training data (normal only, after cleaning) |
| pilot_data.csv | 3,087 | Validation data with 2 real anomaly events |
| test.csv + test_labels.csv | 42 (36 normal, 6 anomaly) | Hold-out test set (never used for training) |

**Important:** The normal_data.csv has a higher humidity baseline (~62%) than pilot/test (~59-60%). This **dataset shift** means a model trained on normal_data may not generalize to deployment conditions matching pilot/test. We'll document this clearly.

---

## SECTION 1: Imports and Configuration

In [ ]:
# Standard libraries
import json
import warnings
from pathlib import Path
from datetime import datetime

# Data science stack
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ML
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix

# Configuration
import yaml

# Plot styling
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

# Suppress harmless warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning, module='nbformat')

print("Imports successful!")
print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")
print(f"matplotlib: {plt.matplotlib.__version__}")
print(f"seaborn: {sns.__version__}")
print(f"scikit-learn: {__import__('sklearn').__version__}")

In [ ]:
# Load configuration from YAML
config_path = Path("config.yaml")
if config_path.exists():
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    print("Configuration loaded from config.yaml")
else:
    # Fallback defaults
    config = {
        "serial": {"port": "COM19", "baudrate": 115200, "timeout": 1.0, "sampling_interval_ms": 2000},
        "data": {"raw_dir": "data/raw", "processed_dir": "data/processed", "features_dir": "data/processed/features"},
        "features": {"rolling_windows": [5, 15], "gap_threshold_seconds": 10},
        "model": {"type": "isolation_forest", "n_estimators": 200, "contamination": 0.01, "random_state": 42}
    }
    print("Using default configuration")

# Convenience variables
RAW_DIR = Path(config["data"]["raw_dir"])
PROCESSED_DIR = Path(config["data"]["processed_dir"])
FEATURES_DIR = Path(config["data"]["features_dir"])
GAP_THRESHOLD = config["features"]["gap_threshold_seconds"]
ROLLING_WINDOWS = config["features"]["rolling_windows"]

print(f"Raw data directory: {RAW_DIR}")
print(f"Processed data directory: {PROCESSED_DIR}")
print(f"Features directory: {FEATURES_DIR}")
print(f"Gap threshold: {GAP_THRESHOLD} seconds")
print(f"Rolling windows: {ROLLING_WINDOWS} observations")

## SECTION 2: Load Original Datasets

### Key Concept: Reading CSV Files
- **`pd.read_csv()`** reads a CSV file into a DataFrame
- By default, it assumes the **first row is the header** (column names)
- If your file has **no header**, use `header=None` and provide names with `names=[...]`
- The **original files in `data/raw/` are never modified** - we only read them

### Our Files:
- `normal_data.csv` - **NO header** (first row is data)
- `pilot_data.csv` - HAS header
- `test.csv` - HAS header
- `test_labels.csv` - HAS header (event-level label, not row-level)

In [ ]:
def load_datasets():
    """
    Load all four raw datasets from data/raw/.
    Returns a dictionary with DataFrames for each dataset.
    """
    print("=== Loading Raw Datasets ===")

    # normal_data.csv has NO header - first row is data
    normal = pd.read_csv(
        RAW_DIR / "normal_data.csv",
        header=None,                    # No header in file
        names=["timestamp", "temperature", "humidity"]  # Provide column names
    )

    # The other files HAVE headers
    pilot = pd.read_csv(RAW_DIR / "pilot_data.csv")
    test = pd.read_csv(RAW_DIR / "test.csv")
    test_labels = pd.read_csv(RAW_DIR / "test_labels.csv")

    # Convert timestamps to datetime
    for df in [normal, pilot, test]:
        df["timestamp"] = pd.to_datetime(df["timestamp"])
    test_labels["start"] = pd.to_datetime(test_labels["start"])
    test_labels["end"] = pd.to_datetime(test_labels["end"])

    # Sort chronologically (stable sort preserves original order for equal timestamps)
    normal = normal.sort_values("timestamp", kind="stable").reset_index(drop=True)
    pilot = pilot.sort_values("timestamp", kind="stable").reset_index(drop=True)
    test = test.sort_values("timestamp", kind="stable").reset_index(drop=True)

    # Quick verification
    print(f"Normal: {len(normal)} rows, {normal['timestamp'].min()} to {normal['timestamp'].max()}")
    print(f"Pilot:  {len(pilot)} rows, {pilot['timestamp'].min()} to {pilot['timestamp'].max()}")
    print(f"Test:   {len(test)} rows, {test['timestamp'].min()} to {test['timestamp'].max()}")
    print(f"Labels: {len(test_labels)} rows")
    print(test_labels)

    return {"normal": normal, "pilot": pilot, "test": test, "test_labels": test_labels}

# Load all datasets
datasets = load_datasets()
normal = datasets["normal"]
pilot = datasets["pilot"]
test = datasets["test"]
test_labels = datasets["test_labels"]

## SECTION 3: Exploratory Data Analysis (EDA)

### What is EDA and Why Do We Need It?

**EDA (Exploratory Data Analysis)** is the process of understanding your data before modeling. It answers:
- What does "normal" look like? (central tendency, spread, range)
- Are there obvious problems? (missing values, duplicates, impossible values)
- What do anomalies look like? (sudden spikes, impossible combinations)
- Are there data quality issues? (gaps, duplicates, sensor errors)

### Key Python Functions We'll Use
- `df.describe()` - computes count, mean, std, min, quartiles, max
- `df.quantile([0.25, 0.5, 0.75])` - explicit quartile calculation
- `df.var()` - variance
- `df.isnull().sum()` - missing values per column
- `df.duplicated().sum()` - exact duplicate rows
- `df['timestamp'].diff().dt.total_seconds()` - time difference between consecutive rows
- `value_counts()` - frequency of each unique value

### Statistical Concepts
- **Mean** - average value
- **Median** - middle value (robust to outliers)
- **Standard Deviation (Std)** - how spread out the data is
- **IQR (Interquartile Range)** = Q3 - Q1, middle 50% spread
- **IQR Fences** - outliers are below Q1-1.5xIQR or above Q3+1.5xIQR
- **Z-Score** = (value - mean) / std, measures how many standard deviations from mean
  - |z| > 3 typically considered outlier (assumes Gaussian)